# BÀI 5: QUÉT THƯ MỤC VÀ GỘP HÀNG LOẠT FILE EXCEL VỚI PATHLIB & TRY...EXCEPT
## Mục tiêu thực hành:
1. Sử dụng thư viện hướng đối tượng `pathlib.Path` để quét tự động tất cả các file Excel trong thư mục và thư mục con.
2. Tự động loại bỏ các file tạm/rác của Windows và Excel (`~$*`).
3. Trích xuất tên đơn vị/chi nhánh trực tiếp từ tên file để làm cột định danh nguồn (`Ten_Don_Vi`).
4. Ứng dụng cơ chế `try...except` để chống sập chương trình khi gặp file lỗi/hỏng, đồng thời ghi nhận báo cáo lỗi.
5. Gộp hàng loạt DataFrame thành một bảng Master Data hoàn chỉnh bằng `pd.concat`.

### Ô 1: Giả lập tạo thư mục và 4 file Excel kiểm kê thực tế
> Khởi tạo thư mục `du_lieu_chi_nhanh/` gồm: 3 file dữ liệu chuẩn từ các phòng giao dịch và 1 file rác/hỏng giả lập để thử nghiệm cơ chế chống lỗi.

In [ ]:
from pathlib import Path
import pandas as pd

# 1. Khởi tạo thư mục thực hành
data_dir = Path("du_lieu_chi_nhanh")
data_dir.mkdir(parents=True, exist_ok=True)

# 2. Dữ liệu chuẩn cho 3 phòng giao dịch
branches_data = {
    "KiemKe_PGD_NinhKieu.xlsx": [
        {"Ma_TS": "NK-001", "Ten_TS": "Máy in HP PGD Ninh Kiều", "So_luong": 2, "Gia_tri": 9000000},
        {"Ma_TS": "NK-002", "Ten_TS": "Máy vi tính Dell PC", "So_luong": 5, "Gia_tri": 22500000}
    ],
    "KiemKe_PGD_CaiRang.xlsx": [
        {"Ma_TS": "CR-001", "Ten_TS": "Máy đếm tiền Glory", "So_luong": 3, "Gia_tri": 15000000},
        {"Ma_TS": "CR-002", "Ten_TS": "Máy scan màu Canon", "So_luong": 1, "Gia_tri": 4500000}
    ],
    "KiemKe_PGD_BinhThuy.xlsx": [
        {"Ma_TS": "BT-001", "Ten_TS": "Bộ phát Wifi chuyên dụng", "So_luong": 4, "Gia_tri": 8000000},
        {"Ma_TS": "BT-002", "Ten_TS": "Máy in tem nhãn Zebra", "So_luong": 2, "Gia_tri": 12000000}
    ]
}

for filename, records in branches_data.items():
    file_path = data_dir / filename
    pd.DataFrame(records).to_excel(file_path, index=False)

# 3. Tạo thêm 1 file rác tạm của Excel (~$...) và 1 file text giả lập bị lỗi định dạng
temp_file = data_dir / "~$KiemKe_PGD_NinhKieu.xlsx"
temp_file.write_text("Day la file lock tam thoi cua Excel", encoding="utf-8")

corrupt_file = data_dir / "KiemKe_PGD_LoiDinhDang.xlsx"
corrupt_file.write_text("Noi dung khong phai file excel", encoding="utf-8")

print(f"Đã tạo môi trường mẫu tại: {data_dir.resolve()}")

Đã tạo môi trường mẫu tại: D:\Python_Project\Python_data_apps\Code_vidu\du_lieu_chi_nhanh


### Ô 2: Quét thư mục bằng `pathlib.Path.glob` & Lọc bỏ file rác `~$*`
> Dùng `Path.glob("*.xlsx")` để tìm file, kết hợp điều kiện `not file.name.startswith("~$")` để loại trừ hoàn toàn các file ẩn/khóa tạm của Excel.

In [2]:
all_excel_files = list(data_dir.glob("*.xlsx"))
print("Tổng số file tìm thấy:", len(all_excel_files))

# Lọc danh sách file hợp lệ (loại bỏ file tạm bắt đầu bằng ~$)
valid_files = [f for f in all_excel_files if not f.name.startswith("~$")]

print(f"Số file hợp lệ cần xử lý: {len(valid_files)}")
for f in valid_files:
    print(" ->", f.name)

Tổng số file tìm thấy: 5
Số file hợp lệ cần xử lý: 4
 -> KiemKe_PGD_BinhThuy.xlsx
 -> KiemKe_PGD_CaiRang.xlsx
 -> KiemKe_PGD_LoiDinhDang.xlsx
 -> KiemKe_PGD_NinhKieu.xlsx


### Ô 3: Vòng lặp nạp dữ liệu hàng loạt kèm cơ chế `try...except` và trích xuất tên đơn vị
> Nếu file bị hỏng hoặc sai định dạng, khối `except` sẽ bắt lại và ghi vào danh sách lỗi mà không dừng chương trình.

In [3]:
dfs = []
error_logs = []

for file_path in valid_files:
    try:
        # Đọc dữ liệu từng file
        df_temp = pd.read_excel(file_path)
        
        # Trích xuất tên đơn vị từ tên file (Ví dụ: KiemKe_PGD_NinhKieu -> PGD_NinhKieu)
        ten_don_vi = file_path.stem.replace("KiemKe_", "")
        df_temp["Don_Vi"] = ten_don_vi
        df_temp["Nguon_File"] = file_path.name
        
        dfs.append(df_temp)
        print(f"[THÀNH CÔNG] Đã nạp {len(df_temp)} dòng từ {file_path.name}")
        
    except Exception as e:
        # Bắt lỗi an toàn, không làm dừng chương trình
        error_logs.append({"File": file_path.name, "Loi": str(e)})
        print(f"[BỎ QUA - LỖI] File hỏng {file_path.name}: {e}")

[THÀNH CÔNG] Đã nạp 2 dòng từ KiemKe_PGD_BinhThuy.xlsx
[THÀNH CÔNG] Đã nạp 2 dòng từ KiemKe_PGD_CaiRang.xlsx
[BỎ QUA - LỖI] File hỏng KiemKe_PGD_LoiDinhDang.xlsx: Excel file format cannot be determined, you must specify an engine manually.
[THÀNH CÔNG] Đã nạp 2 dòng từ KiemKe_PGD_NinhKieu.xlsx


### Ô 4: Hợp nhất thành Master Data (`pd.concat`) & Báo cáo file lỗi
> Xếp chồng toàn bộ DataFrame thành một bảng dữ liệu duy nhất và kiểm tra đối soát file lỗi.

In [4]:
if dfs:
    # Gộp toàn bộ dữ liệu hợp lệ
    df_master = pd.concat(dfs, ignore_index=True)
    print(f"--- HỢP NHẤT HOÀN TẤT: {len(df_master)} DÒNG DỮ LIỆU ---")
    display(df_master)
else:
    print("Không có dữ liệu hợp lệ nào được nạp!")

if error_logs:
    df_errors = pd.DataFrame(error_logs)
    print("\n--- DANH SÁCH CÁC FILE LỖI ĐÃ TỰ ĐỘNG BỎ QUA ---")
    display(df_errors)

--- HỢP NHẤT HOÀN TẤT: 6 DÒNG DỮ LIỆU ---


,Ma_TS,Ten_TS,So_luong,Gia_tri,Don_Vi,Nguon_File
0,BT-001,Bộ phát Wifi chuyên dụng,4,8000000,PGD_BinhThuy,KiemKe_PGD_BinhThuy.xlsx
1,BT-002,Máy in tem nhãn Zebra,2,12000000,PGD_BinhThuy,KiemKe_PGD_BinhThuy.xlsx
2,CR-001,Máy đếm tiền Glory,3,15000000,PGD_CaiRang,KiemKe_PGD_CaiRang.xlsx
3,CR-002,Máy scan màu Canon,1,4500000,PGD_CaiRang,KiemKe_PGD_CaiRang.xlsx
4,NK-001,Máy in HP PGD Ninh Kiều,2,9000000,PGD_NinhKieu,KiemKe_PGD_NinhKieu.xlsx
5,NK-002,Máy vi tính Dell PC,5,22500000,PGD_NinhKieu,KiemKe_PGD_NinhKieu.xlsx



--- DANH SÁCH CÁC FILE LỖI ĐÃ TỰ ĐỘNG BỎ QUA ---


,File,Loi
0,KiemKe_PGD_LoiDinhDang.xlsx,"Excel file format cannot be determined, you mu..."


### Ô 5: Xuất Master Data ra Excel hoàn thiện với OpenPyXL
> Vận dụng kỹ năng Bài 4: Tự động kẻ bảng, căn lề và định dạng số tiền cho toàn bộ dữ liệu hợp nhất.

In [ ]:
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils import get_column_letter

output_dir = Path("xlsx")
output_dir.mkdir(exist_ok=True)  # Tạo thư mục nếu chưa có
output_master = output_dir / "Master_KiemKe_HopNhat_2026.xlsx"

with pd.ExcelWriter(output_master, engine="openpyxl") as writer:
    df_master.to_excel(writer, sheet_name="Master_Data", index=False)
    if error_logs:
        df_errors.to_excel(writer, sheet_name="Danh_Sach_File_Loi", index=False)
        
    ws = writer.sheets["Master_Data"]
    
    # Header
    header_fill = PatternFill(start_color="00685E", end_color="00685E", fill_type="solid")
    header_font = Font(name="Arial", size=10, bold=True, color="FFFFFF")
    ws.row_dimensions[1].height = 26
    
    for col_idx in range(1, len(df_master.columns) + 1):
        cell = ws.cell(row=1, column=col_idx)
        cell.fill = header_fill
        cell.font = header_font
        cell.alignment = Alignment(horizontal="center", vertical="center")
        
    # Auto-fit cột
    for col in ws.columns:
        col_letter = get_column_letter(col[0].column)
        max_len = max(len(str(cell.value or '')) for cell in col)
        ws.column_dimensions[col_letter].width = max(max_len + 4, 12)

print(f"Đã xuất bảng Master Data thành công ra tệp: {output_master}")

Đã xuất bảng Master Data thành công ra tệp: xlsx\Master_KiemKe_HopNhat_2026.xlsx
